# 📖 Notebook 3: Backup Strategies

## Why This Matters

Replication protects you from hardware failure, but NOT from:
- **Accidental DELETE** — someone runs `DELETE FROM orders` without a WHERE clause
- **Ransomware** — malicious encryption of your data (replicated to standby!)
- **Logical corruption** — a bug writes bad data to the database
- **Compliance** — regulations require you to keep historical backups

**Replication is NOT a backup.** If you delete data on the primary,
the delete is replicated to the standby. You need actual backups.

## Learning Objectives

- Understand full, incremental, and differential backup types
- Perform a logical backup with `pg_dump`
- Perform a physical backup with `pg_basebackup`
- Test backup restoration (the most neglected practice!)
- Understand point-in-time recovery (PITR) concepts

## 🛠️ Setup

```bash
cd enterprise-patterns/bcdr
docker-compose down -v && docker-compose up -d
```

This resets the environment to a clean state.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).

In [1]:
import psycopg2
import subprocess
import time
import os
from tabulate import tabulate

DB_PRIMARY = {
    "host": "localhost", "port": 5432,
    "database": "bcdr_demo", "user": "demo", "password": "demo"
}

def get_primary_connection():
    return psycopg2.connect(**DB_PRIMARY)

def docker_exec(container, cmd):
    result = subprocess.run(
        ["docker", "exec", container] + cmd,
        capture_output=True, text=True, timeout=60
    )
    return result.stdout.strip(), result.stderr.strip()

# Test connection
conn = get_primary_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM orders")
print(f"✅ Connected. Orders in database: {cur.fetchone()[0]}")
conn.close()

✅ Connected. Orders in database: 500


## 📚 Backup Types Explained

### Full Backup
- Copies **everything** in the database
- Slowest to create, largest file size
- Fastest to restore (just load one file)
- **Example**: `pg_dump` or `pg_basebackup`

### Incremental Backup
- Copies only data that changed **since the last backup of any type**
- Fastest to create, smallest file size
- Slowest to restore (need full + every incremental in order)
- **Example**: WAL archiving between `pg_basebackup` runs

### Differential Backup
- Copies only data that changed **since the last full backup**
- Medium speed to create, medium file size
- Medium restore speed (need full + latest differential)

```
         Day 1       Day 2       Day 3       Day 4       Day 5
Full:   [AAABBB]
Incr:               [CC]        [DD]        [EE]        [FF]
Diff:               [CC]        [CCDD]      [CCDDEE]    [CCDDEEFF]

To restore Day 5:
  Full only:  Not possible (Day 1 data only)
  Incremental: Full + CC + DD + EE + FF (5 files)
  Differential: Full + CCDDEEFF (2 files)
```

In [2]:
# =============================================================================
# Demo: Logical Backup with pg_dump
# =============================================================================
# pg_dump creates a SQL script that recreates your database.
# It is a LOGICAL backup — human-readable SQL statements.

print("=" * 65)
print("LOGICAL BACKUP WITH pg_dump")
print("=" * 65)

# Create a full logical backup
start_time = time.time()
out, err = docker_exec('bcdr-pg-primary', [
    "pg_dump",
    "-U", "demo",
    "-d", "bcdr_demo",
    "--format=custom",       # compressed binary format
    "--file=/tmp/backup.dump"
])
backup_time = time.time() - start_time

if err and 'error' in err.lower():
    print(f"❌ Backup failed: {err}")
else:
    print(f"  ✅ Backup completed in {backup_time:.2f} seconds")

    # Check backup size
    out, _ = docker_exec('bcdr-pg-primary', [
        "ls", "-lh", "/tmp/backup.dump"
    ])
    print(f"  📦 Backup file: {out}")

    # Verify backup contents (list what is inside)
    out, _ = docker_exec('bcdr-pg-primary', [
        "pg_restore", "--list", "/tmp/backup.dump"
    ])
    lines = out.strip().split('\n')
    print(f"  📋 Backup contains {len(lines)} objects")
    print("\n  First 10 objects in backup:")
    for line in lines[:10]:
        print(f"    {line}")

LOGICAL BACKUP WITH pg_dump
  ✅ Backup completed in 0.14 seconds


  📦 Backup file: -rw-r--r-- 1 root root 62K Apr 20 19:20 /tmp/backup.dump
  📋 Backup contains 63 objects

  First 10 objects in backup:
    ;
    ; Archive created at 2026-04-20 19:20:10 UTC
    ;     dbname: bcdr_demo
    ;     TOC Entries: 52
    ;     Compression: gzip
    ;     Dump Version: 1.15-0
    ;     Format: CUSTOM
    ;     Integer: 4 bytes
    ;     Offset: 8 bytes
    ;     Dumped from database version: 16.13 (Debian 16.13-1.pgdg13+1)


## 📚 Physical Backup with pg_basebackup

`pg_basebackup` creates a **physical copy** of the entire database cluster.
This is the same tool we used to set up our standby server!

### Logical vs Physical Backup

| Feature | Logical (pg_dump) | Physical (pg_basebackup) |
|---------|------------------|-------------------------|
| What it copies | Table data + schema | Entire data directory (binary) |
| Speed | Slower (reads every row) | Faster (copies files) |
| Selective restore | Yes (individual tables) | No (all or nothing) |
| Cross-version | Yes (can restore to newer PG) | No (same major version only) |
| Point-in-time recovery | No | Yes (with WAL archiving) |
| Best for | Small databases, migrations | Large databases, PITR |

In [3]:
# =============================================================================
# Demo: Physical Backup with pg_basebackup
# =============================================================================

print("=" * 65)
print("PHYSICAL BACKUP WITH pg_basebackup")
print("=" * 65)

start_time = time.time()
out, err = docker_exec('bcdr-pg-primary', [
    "pg_basebackup",
    "-U", "demo",
    "-D", "/tmp/physical_backup",
    "-Ft",    # tar format
    "-z",     # gzip compression
    "-Xs",    # stream WAL during backup
    "-P"      # show progress
])
backup_time = time.time() - start_time

print(f"  Time: {backup_time:.2f} seconds")
if err:
    # pg_basebackup outputs progress to stderr
    progress_lines = [l for l in err.split('\n') if '%' in l]
    if progress_lines:
        print(f"  Progress: {progress_lines[-1].strip()}")

# Check backup files
out, _ = docker_exec('bcdr-pg-primary', [
    "ls", "-lh", "/tmp/physical_backup/"
])
print(f"\n  Backup files:")
for line in out.split('\n'):
    if line.strip():
        print(f"    {line.strip()}")

print("\n💡 base.tar.gz = all database files, pg_wal.tar.gz = WAL logs")
print("   Together these can restore the database to this exact point in time.")

PHYSICAL BACKUP WITH pg_basebackup


  Time: 6.38 seconds
  Progress: 31686/31686 kB (100%), 1/1 tablespace

  Backup files:
    total 4.4M
    -rw------- 1 root root 182K Apr 20 19:20 backup_manifest
    -rw------- 1 root root 4.2M Apr 20 19:20 base.tar.gz
    -rw------- 1 root root  17K Apr 20 19:20 pg_wal.tar.gz

💡 base.tar.gz = all database files, pg_wal.tar.gz = WAL logs
   Together these can restore the database to this exact point in time.


## 📚 Backup Verification — The Most Neglected Practice

> **An untested backup is not a backup — it is a hope.**

Many organizations discover their backups are corrupted or incomplete
only when they need to restore them during a real disaster.

### Backup Verification Checklist

1. **Can you restore it?** — Actually restore to a test database
2. **Is the data complete?** — Compare row counts, checksums
3. **How long does restore take?** — This is your actual RTO for backup-based recovery
4. **Can someone else do it?** — Document the procedure, test with different team members
5. **Is the backup accessible?** — Can you reach it during a disaster?

In [4]:
# =============================================================================
# Demo: Restore and Verify a Backup
# =============================================================================
# We will restore our pg_dump backup to a different database and verify it.

print("=" * 65)
print("BACKUP VERIFICATION: Restore + Verify")
print("=" * 65)

# Step 1: Create a test database for restoration
print("\nStep 1: Creating test database for restore...")
docker_exec('bcdr-pg-primary', [
    "psql", "-U", "demo", "-d", "postgres",
    "-c", "DROP DATABASE IF EXISTS bcdr_restore_test"
])
docker_exec('bcdr-pg-primary', [
    "psql", "-U", "demo", "-d", "postgres",
    "-c", "CREATE DATABASE bcdr_restore_test"
])
print("  ✅ Test database created")

# Step 2: Restore the backup
print("\nStep 2: Restoring backup...")
start_time = time.time()
out, err = docker_exec('bcdr-pg-primary', [
    "pg_restore",
    "-U", "demo",
    "-d", "bcdr_restore_test",
    "--no-owner",
    "/tmp/backup.dump"
])
restore_time = time.time() - start_time
print(f"  ✅ Restore completed in {restore_time:.2f} seconds")

# Step 3: Verify data integrity
print("\nStep 3: Verifying data integrity...")

conn_orig = get_primary_connection()
cur_orig = conn_orig.cursor()

conn_rest = psycopg2.connect(
    host="localhost", port=5432,
    database="bcdr_restore_test", user="demo", password="demo"
)
cur_rest = conn_rest.cursor()

tables = ['customers', 'orders', 'order_items', 'payments', 'audit_log']
table_data = []
all_match = True

for tbl in tables:
    cur_orig.execute(f"SELECT COUNT(*) FROM {tbl}")
    orig_count = cur_orig.fetchone()[0]
    cur_rest.execute(f"SELECT COUNT(*) FROM {tbl}")
    rest_count = cur_rest.fetchone()[0]
    match = "✅" if orig_count == rest_count else "❌"
    if orig_count != rest_count:
        all_match = False
    table_data.append([tbl, orig_count, rest_count, match])

print(tabulate(table_data,
    headers=["Table", "Original", "Restored", "Match"],
    tablefmt="grid"))

if all_match:
    print("\n🎉 All row counts match! Backup is verified.")
else:
    print("\n⚠️  Row count mismatch detected!")

print(f"\n📊 Restore time: {restore_time:.2f}s — this is your backup-based RTO")

conn_orig.close()
conn_rest.close()

# Cleanup
docker_exec('bcdr-pg-primary', [
    "psql", "-U", "demo", "-d", "postgres",
    "-c", "DROP DATABASE IF EXISTS bcdr_restore_test"
])

BACKUP VERIFICATION: Restore + Verify

Step 1: Creating test database for restore...


  ✅ Test database created

Step 2: Restoring backup...


  ✅ Restore completed in 0.12 seconds

Step 3: Verifying data integrity...
+-------------+------------+------------+---------+
| Table       |   Original |   Restored | Match   |
+=============+============+============+=========+
| customers   |         50 |         50 | ✅      |
+-------------+------------+------------+---------+
| orders      |        500 |        500 | ✅      |
+-------------+------------+------------+---------+
| order_items |       1500 |       1500 | ✅      |
+-------------+------------+------------+---------+
| payments    |        500 |        500 | ✅      |
+-------------+------------+------------+---------+
| audit_log   |          0 |          0 | ✅      |
+-------------+------------+------------+---------+

🎉 All row counts match! Backup is verified.

📊 Restore time: 0.12s — this is your backup-based RTO


('DROP DATABASE', '')

## 📚 Point-in-Time Recovery (PITR)

PITR lets you restore your database to **any specific moment in time**.

```
                  pg_basebackup    accidental    restore
  ─────────────────[BACKUP]─────────[DELETE]──────[HERE]──────
                     ▲                              ▲
                     │     WAL logs fill the gap    │
                     │◄────────────────────────────►│
```

### How It Works

1. Start with a `pg_basebackup` (your base snapshot)
2. PostgreSQL continuously archives WAL files (the change log)
3. To recover: restore the base backup, then replay WAL files up to your target time
4. This gets you to any point between the backup and the disaster

### Requirements
- `archive_mode = on` in PostgreSQL config (we have this!)
- WAL archive storage that survives the disaster
- A known good timestamp to recover to

In [5]:
# =============================================================================
# Demo: Check WAL Archiving Status
# =============================================================================

conn = get_primary_connection()
cur = conn.cursor()

# Check archive settings
cur.execute(
    "SELECT name, setting FROM pg_settings "
    "WHERE name IN ('archive_mode', 'archive_command', 'wal_level') "
    "ORDER BY name"
)
settings = cur.fetchall()

print("=" * 65)
print("WAL ARCHIVE CONFIGURATION")
print("=" * 65)
for name, value in settings:
    print(f"  {name}: {value}")

# Check archived files
cur.execute(
    "SELECT archived_count, failed_count, "
    "last_archived_wal, last_archived_time "
    "FROM pg_stat_archiver"
)
row = cur.fetchone()
print(f"\n  Archived WAL files: {row[0]}")
print(f"  Failed archives:    {row[1]}")
print(f"  Last archived WAL:  {row[2]}")
print(f"  Last archive time:  {row[3]}")

conn.close()

print("\n💡 WAL archiving + pg_basebackup = point-in-time recovery capability.")
print("   This is how enterprises achieve RPO of minutes or even seconds.")

WAL ARCHIVE CONFIGURATION
  archive_command: test ! -f /var/lib/postgresql/archive/%f && cp %p /var/lib/postgresql/archive/%f
  archive_mode: on
  wal_level: replica

  Archived WAL files: 0
  Failed archives:    15
  Last archived WAL:  None
  Last archive time:  None

💡 WAL archiving + pg_basebackup = point-in-time recovery capability.
   This is how enterprises achieve RPO of minutes or even seconds.


## 📚 The 3-2-1 Backup Rule

A time-tested rule used by IT departments everywhere:

```
   3 copies of your data
   └─ on 2 different types of media
      └─ with 1 copy off-site
```

### Why each number matters

| Number | What | Why |
|--------|------|-----|
| **3 copies** | Your live data + at least 2 backups | Any single copy can fail or be corrupted |
| **2 media types** | e.g., local disk + cloud object storage | Protects against media-specific failures |
| **1 off-site** | In a different building / region / account | Protects against fire, flood, ransomware, rogue admin |

### Modern version: 3-2-1-1-0

Teams add two extras for ransomware defense:
- **1 copy immutable** (write-once, cannot be deleted for N days)
- **0 errors** — every backup is verified by automated restore tests

### What this lab simulates vs. real life

| Layer | This lab | Production BCDR |
|-------|----------|-----------------|
| Live DB | `bcdr-pg-primary` | Primary DB cluster |
| Hot copy | `bcdr-pg-standby` | Replica in another AZ |
| Local backup | `/tmp/backup.dump` in the container | Backup server in the same DC |
| Off-site backup | ⚠️ not simulated | S3 / GCS / Azure Blob in a **different region** |

## 📚 Hands-On: "Replication is NOT a Backup"

This is the single most common BCDR mistake. Let us **prove** it.

### The scenario

1. A developer runs `DELETE FROM orders` without a `WHERE` clause.
2. The delete is **immediately replicated** to the standby.
3. The standby cannot save us — it has the same deleted state.
4. Only a real backup (taken BEFORE the disaster) can bring the data back.

### Bad → Better → Best

| Approach | Outcome |
|----------|---------|
| ❌ **Bad**: "We have a replica, that is our backup" | Data is gone on both primary and replica — unrecoverable |
| ⚠️ **Better**: Nightly `pg_dump` on the same server | Can recover, but up to 24h of data loss, and host-level failure takes the backup too |
| ✅ **Best**: Frequent backups + WAL archiving + off-site (3-2-1) | Recover to any point in time, survive fires and ransomware |

In [6]:
# =============================================================================
# Demo: Replication is NOT a Backup
# =============================================================================
# We will: take a backup, delete all orders, confirm the delete is replicated
# to the standby, then restore from the backup.

print('=' * 65)
print("DEMO: 'Replication is NOT a Backup'")
print('=' * 65)

# Step 1: take a fresh backup BEFORE the disaster.
print('\nStep 1: Taking a pg_dump backup (our safety net)...')
docker_exec('bcdr-pg-primary', [
    'pg_dump', '-U', 'demo', '-d', 'bcdr_demo',
    '--format=custom', '--file=/tmp/pre_disaster.dump',
])
print('  ✅ Backup saved to /tmp/pre_disaster.dump')

# Open connections. Use autocommit from the start so set_session is safe.
pconn = get_primary_connection()
pconn.autocommit = True
pcur = pconn.cursor()

sconn = psycopg2.connect(host='localhost', port=5433,
                         database='bcdr_demo', user='demo', password='demo')
sconn.autocommit = True
scur = sconn.cursor()

# Step 2: record order counts on primary + standby
pcur.execute('SELECT COUNT(*) FROM orders'); p_before = pcur.fetchone()[0]
scur.execute('SELECT COUNT(*) FROM orders'); s_before = scur.fetchone()[0]
print(f'\nStep 2: Before disaster → primary: {p_before} orders, standby: {s_before} orders')

# Step 3: the 'disaster' — developer deletes everything.
print('\nStep 3: 💥 Running "DELETE FROM orders" on PRIMARY...')
pcur.execute('DELETE FROM payments')     # child rows first (FK)
pcur.execute('DELETE FROM order_items')
pcur.execute('DELETE FROM orders')
time.sleep(1)  # let replication catch up

pcur.execute('SELECT COUNT(*) FROM orders'); p_after = pcur.fetchone()[0]
scur.execute('SELECT COUNT(*) FROM orders'); s_after = scur.fetchone()[0]
print(f'  After DELETE → primary: {p_after} orders, standby: {s_after} orders')
print('  ❌ The standby was NOT a safety net — the DELETE replicated instantly.')

# Step 4: restore from the backup we took in step 1.
# --data-only restores only rows (schema already exists).
print('\nStep 4: Restoring from pre-disaster backup...')
docker_exec('bcdr-pg-primary', [
    'pg_restore', '-U', 'demo', '-d', 'bcdr_demo',
    '--data-only', '--disable-triggers',
    '/tmp/pre_disaster.dump',
])
pcur.execute('SELECT COUNT(*) FROM orders'); p_recovered = pcur.fetchone()[0]
print(f'  ✅ Primary after restore: {p_recovered} orders')

print('\n💡 Takeaways:')
print('   • Replication protects against HARDWARE failure, not LOGICAL errors.')
print('   • Always keep real backups — and keep some OFF-SITE (3-2-1 rule).')
print('   • For near-zero RPO on logical errors, enable point-in-time recovery.')

pconn.close(); sconn.close()


DEMO: 'Replication is NOT a Backup'

Step 1: Taking a pg_dump backup (our safety net)...


  ✅ Backup saved to /tmp/pre_disaster.dump

Step 2: Before disaster → primary: 500 orders, standby: 500 orders

Step 3: 💥 Running "DELETE FROM orders" on PRIMARY...


  After DELETE → primary: 0 orders, standby: 0 orders
  ❌ The standby was NOT a safety net — the DELETE replicated instantly.

Step 4: Restoring from pre-disaster backup...
  ✅ Primary after restore: 500 orders

💡 Takeaways:
   • Replication protects against HARDWARE failure, not LOGICAL errors.
   • Always keep real backups — and keep some OFF-SITE (3-2-1 rule).
   • For near-zero RPO on logical errors, enable point-in-time recovery.


## 📝 Summary

### What You Learned

1. **Replication is NOT backup** — Deletes and corruption replicate too!
2. **Full/Incremental/Differential** — Trade-offs between backup speed, size, and restore speed.
3. **pg_dump** — Logical backup (SQL). Good for small DBs, selective restore, cross-version.
4. **pg_basebackup** — Physical backup (binary). Good for large DBs and PITR.
5. **Backup verification** — Always restore and verify. Measure your actual restore time.
6. **PITR** — Base backup + WAL archive = restore to any point in time.

### Key Takeaway

> **An untested backup is not a backup. Schedule regular restore tests.**

### Next Notebook

In **Notebook 4**, we run a full disaster recovery drill — simulating
a primary failure and measuring our actual RTO.